# 11 — Version 2 LightGBM

**Owners:** Saravana / Nebal

This is the first Version 2 experiment because it is the fastest reliable way to
validate the new behavioural features. It uses three expanding time folds and
never selects parameters from the final test period.


## What Version 2 changes—and why

Version 1 remains our reproducible baseline. Version 2 adds behaviour that the
winning Kaggle solution showed was valuable, but implements it in a stricter
real-time form:

- `D` values are normalized against transaction day to expose stable date anchors.
- a conservative `uid_proxy` describes a possible client without using it as a label;
- counts, time since previous use, amount history and unique-value history describe
  behaviour;
- every historical feature uses only earlier transactions; and
- no feature reads `isFraud`, later rows, validation labels, or test labels.

The newest 15% remains the final test period. It is never used for feature or
hyperparameter selection.


In [ ]:
from pathlib import Path
_start = Path.cwd().resolve()
for _candidate in [_start, *_start.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

V2_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "v2"
V2_ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v2"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Project root:", PROJECT_ROOT)
print("Version 2 data:", V2_DATA_DIR)
print("Version 2 artifacts:", V2_ARTIFACT_ROOT)


In [ ]:
required = [V2_DATA_DIR / name for name in ["train.parquet", "validation.parquet", "test.parquet"]]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run 10_v2_behavioral_data_preparation.ipynb first. Missing: " + ", ".join(missing)
    )

train, validation, test = [pd.read_parquet(path) for path in required]
FAST_RUN = False  # Only for code checks. Never report FAST_RUN metrics.
if FAST_RUN:
    def debug_sample(frame, rows):
        return (frame.groupby("isFraud", group_keys=False)
                .apply(lambda group: group.sample(
                    n=max(1, round(rows * len(group) / len(frame))),
                    random_state=RANDOM_SEED), include_groups=True)
                .sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True))
    train = debug_sample(train, 60_000)
    validation = debug_sample(validation, 20_000)
    test = debug_sample(test, 20_000)

TARGET = "isFraud"
DROP_FROM_MODEL = ["isFraud", "TransactionID"]
X_train, y_train = train.drop(columns=DROP_FROM_MODEL), train[TARGET].astype("int8")
X_validation, y_validation = validation.drop(columns=DROP_FROM_MODEL), validation[TARGET].astype("int8")
X_test, y_test = test.drop(columns=DROP_FROM_MODEL), test[TARGET].astype("int8")
development = pd.concat([train, validation], ignore_index=True)
print("Train:", X_train.shape, "fraud rate:", f"{y_train.mean():.4%}")
print("Validation:", X_validation.shape, "fraud rate:", f"{y_validation.mean():.4%}")
print("Test:", X_test.shape, "fraud rate:", f"{y_test.mean():.4%}")


In [ ]:
MODEL_KEY = "lightgbm"

from datetime import datetime, timezone
from src.fraud_pipeline.artifacts import build_manifest, package_versions, write_json
from src.fraud_pipeline.evaluation import evaluate_binary_classifier, select_operating_threshold

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = V2_ARTIFACT_ROOT / MODEL_KEY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("This Version 2 run will be saved to:", RUN_DIR)


## Candidate configurations

We deliberately test a small, explainable search rather than hundreds of trials.
Class weight is included because full inverse-frequency weighting is not always
best for ranking rare fraud.


In [ ]:
import joblib, lightgbm as lgb
from sklearn.metrics import average_precision_score
from src.fraud_pipeline.preprocessing import LightGBMPreprocessor
from src.fraud_pipeline.validation_v2 import expanding_time_folds, positive_weight

candidates = [
    {"name": "balanced_64", "num_leaves": 64, "min_child_samples": 50,
     "weight_mode": "balanced", "reg_alpha": 0.1, "reg_lambda": 1.0},
    {"name": "sqrt_96", "num_leaves": 96, "min_child_samples": 100,
     "weight_mode": "sqrt_balanced", "reg_alpha": 0.2, "reg_lambda": 3.0},
    {"name": "unweighted_48", "num_leaves": 48, "min_child_samples": 100,
     "weight_mode": "none", "reg_alpha": 0.2, "reg_lambda": 5.0},
]
if FAST_RUN:
    candidates = candidates[:1]


## Rolling time validation


In [ ]:
development = development.sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True)
X_dev = development.drop(columns=DROP_FROM_MODEL)
y_dev = development[TARGET].astype("int8")
folds = expanding_time_folds(len(development))
cv_rows = []

for candidate in candidates:
    for fold_number, (train_index, valid_index) in enumerate(folds, start=1):
        X_fold_train, y_fold_train = X_dev.iloc[train_index], y_dev.iloc[train_index]
        X_fold_valid, y_fold_valid = X_dev.iloc[valid_index], y_dev.iloc[valid_index]
        preprocessor = LightGBMPreprocessor(rare_min_count=20).fit(X_fold_train)
        fold_train = preprocessor.transform(X_fold_train)
        fold_valid = preprocessor.transform(X_fold_valid)
        weight = positive_weight(y_fold_train, candidate["weight_mode"])
        model = lgb.LGBMClassifier(
            objective="binary", metric="None", n_estimators=10_000,
            learning_rate=0.03, num_leaves=candidate["num_leaves"], max_depth=-1,
            min_child_samples=candidate["min_child_samples"], subsample=0.8,
            subsample_freq=1, colsample_bytree=0.8,
            reg_alpha=candidate["reg_alpha"], reg_lambda=candidate["reg_lambda"],
            scale_pos_weight=weight, random_state=RANDOM_SEED, n_jobs=-1,
        )
        started = time.perf_counter()
        model.fit(
            fold_train, y_fold_train,
            eval_set=[(fold_valid, y_fold_valid)], eval_metric="average_precision",
            categorical_feature=preprocessor.categorical_features,
            callbacks=[lgb.early_stopping(300, first_metric_only=True), lgb.log_evaluation(250)],
        )
        probability = model.predict_proba(fold_valid, num_iteration=model.best_iteration_)[:, 1]
        cv_rows.append({
            **candidate, "fold": fold_number, "train_rows": len(train_index),
            "validation_rows": len(valid_index), "best_iteration": model.best_iteration_,
            "validation_pr_auc": average_precision_score(y_fold_valid, probability),
            "seconds": time.perf_counter() - started,
        })
        del preprocessor, model, fold_train, fold_valid, probability
        gc.collect()

cv_results = pd.DataFrame(cv_rows)
display(cv_results)
summary = cv_results.groupby("name")["validation_pr_auc"].agg(["mean", "std"]).sort_values("mean", ascending=False)
display(summary)
best_name = summary.index[0]
best_candidate = next(item for item in candidates if item["name"] == best_name)
print("Selected from rolling validation:", best_candidate)


## Final fit and untouched test evaluation

Final model training still uses the original earliest 70%, next 15% for early
stopping, and latest 15% once for reporting.


In [ ]:
preprocessor = LightGBMPreprocessor(rare_min_count=20).fit(X_train)
X_train_model = preprocessor.transform(X_train)
X_validation_model = preprocessor.transform(X_validation)
weight = positive_weight(y_train, best_candidate["weight_mode"])
model = lgb.LGBMClassifier(
    objective="binary", metric="None", n_estimators=12_000, learning_rate=0.03,
    num_leaves=best_candidate["num_leaves"], max_depth=-1,
    min_child_samples=best_candidate["min_child_samples"], subsample=0.8,
    subsample_freq=1, colsample_bytree=0.8,
    reg_alpha=best_candidate["reg_alpha"], reg_lambda=best_candidate["reg_lambda"],
    scale_pos_weight=weight, random_state=RANDOM_SEED, n_jobs=-1,
)
started = time.perf_counter()
model.fit(
    X_train_model, y_train, eval_set=[(X_validation_model, y_validation)],
    eval_metric="average_precision", categorical_feature=preprocessor.categorical_features,
    callbacks=[lgb.early_stopping(300, first_metric_only=True), lgb.log_evaluation(250)],
)
training_seconds = time.perf_counter() - started
validation_probability = model.predict_proba(X_validation_model, num_iteration=model.best_iteration_)[:, 1]
threshold_record = select_operating_threshold(y_validation, validation_probability, minimum_precision=0.10)
threshold = float(threshold_record["threshold"])
validation_metrics = evaluate_binary_classifier(y_validation, validation_probability, threshold)
del X_train_model
gc.collect()
X_test_model = preprocessor.transform(X_test)
test_probability = model.predict_proba(X_test_model, num_iteration=model.best_iteration_)[:, 1]
test_metrics = evaluate_binary_classifier(y_test, test_probability, threshold)
display(pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])[
    ["pr_auc", "roc_auc", "precision", "recall", "f1", "brier_score"]])


## Save, reload, and package


In [ ]:
importance = pd.DataFrame({
    "feature": model.booster_.feature_name(),
    "gain": model.booster_.feature_importance(importance_type="gain"),
    "split": model.booster_.feature_importance(importance_type="split"),
}).sort_values("gain", ascending=False)
importance.head(150).to_csv(RUN_DIR / "feature_importance.csv", index=False)
cv_results.to_csv(RUN_DIR / "cv_results.csv", index=False)
model.booster_.save_model(str(RUN_DIR / "model.txt"), num_iteration=model.best_iteration_)
joblib.dump(preprocessor, RUN_DIR / "preprocessor.joblib", compress=3)
pd.DataFrame({"TransactionID": validation.TransactionID, "isFraud": y_validation,
              "probability": validation_probability}).to_parquet(RUN_DIR / "validation_predictions.parquet", index=False)
pd.DataFrame({"TransactionID": test.TransactionID, "isFraud": y_test,
              "probability": test_probability}).to_parquet(RUN_DIR / "test_predictions.parquet", index=False)
write_json(RUN_DIR / "threshold.json", threshold_record)
write_json(RUN_DIR / "metrics.json", {"validation": validation_metrics, "test": test_metrics})
write_json(RUN_DIR / "feature_schema.json", {"groups": preprocessor.groups,
           "categorical_features": preprocessor.categorical_features,
           "behavioral_contract": "data/processed/v2/behavioral_contract.json"})
write_json(RUN_DIR / "training_config.json", {
    "model": "v2_lightgbm", "run_id": RUN_ID, "fast_run": FAST_RUN,
    "random_seed": RANDOM_SEED, "training_seconds": training_seconds,
    "best_iteration": model.best_iteration_, "selected_candidate": best_candidate,
    "scale_pos_weight": weight, "parameters": model.get_params(),
    "versions": package_versions(["numpy", "pandas", "scikit-learn", "lightgbm", "joblib"]),
})
loaded_preprocessor = joblib.load(RUN_DIR / "preprocessor.joblib")
loaded_model = lgb.Booster(model_file=str(RUN_DIR / "model.txt"))
sample = loaded_preprocessor.transform(X_validation.iloc[:5])
np.testing.assert_allclose(
    validation_probability[:5], loaded_model.predict(sample), rtol=1e-6, atol=1e-8)


In [ ]:
import shutil
write_json(RUN_DIR / "manifest.json", build_manifest(RUN_DIR))
archive_base = RUN_DIR.parent / f"{MODEL_KEY}_{RUN_ID}"
archive_path = Path(shutil.make_archive(str(archive_base), "gztar", root_dir=RUN_DIR))
print("Reload check passed.")
print("Artifact folder:", RUN_DIR)
print("Share this archive:", archive_path)
